# Práctica 2.2: Clasificación Avanzada y Regresión Supervisada
**Módulo Profesional:** Sistemas de Aprendizaje Automático (Código 5072)  
**Especialización:** Inteligencia Artificial y Big Data (RD 279/2021)  
**Unidad Didáctica 2:** Caracterización de sistemas de aprendizaje automático (Puntos 2.4.1 y 2.4.2 de Marcombo)  
**Criterios de Evaluación Cubiertos:** **CE 2.b** (Tipos y usos), **CE 2.c** (Técnicas y herramientas) y **CE 2.e** (Asociación de modelos a problemas).

---

### Objetivos de la Práctica:
1. Comprender la diferencia entre problemas de **Regresión Continua** ($y \in \mathbb{R}$) y **Clasificación Categórica** ($y \in \{0, 1\}$).
2. Implementar y evaluar modelos de **Regresión Lineal Simple y Múltiple**, midiendo su rendimiento mediante $RMSE$, $MAE$ y $R^2$.
3. Aplicar **Regresión Logística** para clasificación probabilística utilizando la función sigmoide/logit.
4. Construir clasificadores de **Máquinas de Vectores de Soporte (SVM)** con hiperplanos de margen máximo y funciones *kernel*.
5. Analizar el impacto de los hiperparámetros de regularización ($C$) en la prevención del sobreajuste (*overfitting*).

𝑅𝑀𝑆𝐸 , 𝑀𝐴𝐸 y 𝑅2 constituyen las medidas de rendimiento y bondad de ajuste fundamentales para modelos de regresión
. Se utilizan para evaluar cuantitativamente cómo de cerca están las predicciones del modelo frente a los valores continuos reales



1.   MAE (Error Absoluto Medio / Mean Absolute Error)


*   **Definición conceptual**: Es el promedio aritmético de las diferencias absolutas entre el valor predicho por el modelo y el valor real
*   **Significado e interpretación**:
Mide la magnitud media del error directamente en las mismas unidades físicas de la variable objetivo

2. RMSE (Raíz del Error Cuadrático Medio / Root Mean Squared Error)

* **Definición conceptual:** Es la raíz cuadrada del promedio de las diferencias al cuadrado entre los valores predichos y los valores reales
* **Significado e interpretación:**
Al igual que el MAE, se expresa en las mismas unidades de la variable que queremos estimar. Sin embargo, al elevar cada residuo al cuadrado antes de promediar, penaliza con mucha mayor dureza los errores grandes

3. $R^2$ (Coeficiente de Determinación / R-Squared)

* **Definición conceptual:** Mide la proporción de la varianza total de la variable objetivo que es explicada por las variables predictoras del modelo

* **Significado e interpretación:**

Es una métrica adimensional y normalizada (generalmente entre 0.0 y 1.0, o del 0% al 100%):
* $R^2 = 1.0$: Ajuste perfecto; el modelo explica el 100% de la variabilidad de los datos.
* $R^2 = 0.0$: El modelo no aporta información predictiva útil; rinde exactamente igual que predecir siempre la media histórica de la variable.
* $R^2 < 0$: Indica un modelo deficiente que comete más error que la simple media en datos no vistos (test).




## Bloque 01: Configuración del Entorno e Importación de Librerías

In [ ]:
# =====================================================================
# PASO 1: CONFIGURACIÓN DEL ENTORNO E IMPORTACIÓN DE LIBRERÍAS
# Práctica 2.2: Clasificación Avanzada y Regresión Supervisada (Unidad 2)
# =====================================================================

# Librería fundamental para el cálculo numérico algebraico y manipulación de vectores/matrices.
import numpy as np

# Librería estándar para la manipulación, limpieza y análisis estructurado de datos en DataFrames.
import pandas as pd

# Módulo principal de generación de gráficos 2D para visualización de datos en Python.
import matplotlib.pyplot as plt

# Librería de visualización estadística basada en Matplotlib para gráficos avanzados y estilizados.
import seaborn as sns

# Datasets integrados de Scikit-Learn:
# - fetch_california_housing: Dataset continuo para problemas de REGRESIÓN (predicción del valor mediano de viviendas).
# - load_breast_cancer: Dataset biomédico binario para problemas de CLASIFICACIÓN (diagnóstico benigno/maligno).
from sklearn.datasets import fetch_california_housing, load_breast_cancer

# Función para dividir el dataset de forma representativa en conjuntos de Entrenamiento (Train) y Test (Evaluación).
from sklearn.model_selection import train_test_split

# Transformador para estandarizar características restando la media y dividiendo por la desviación típica (Z-score).
# Imprescindible para algoritmos sensibles a la escala de las variables como las Máquinas de Vectores de Soporte (SVM).
from sklearn.preprocessing import StandardScaler

# Modelos Lineales de Scikit-Learn:
# - LinearRegression: Algoritmo supervisado para la predicción de variables cuantitativas continuas (y ∈ ℝ).
# - LogisticRegression: Algoritmo de clasificación categórica binaria basado en la función sigmoide/logit.
from sklearn.linear_model import LinearRegression, LogisticRegression

# Máquinas de Vectores de Soporte (Support Vector Machines):
# - SVC (Support Vector Classifier): Clasificador basado en el cálculo del hiperplano de margen máximo.
# - SVR (Support Vector Regressor): Modelo de regresión basado en márgenes de tolerancia (ε-insensitive).
from sklearn.svm import SVC, SVR

# Métricas cuantitativas para evaluar el rendimiento de los modelos:
# - Métricas de Regresión: Error Cuadrático Medio (MSE), Error Absoluto Medio (MAE) y Coeficiente de Determinación (R²).
# - Métricas de Clasificación: Exactitud (Accuracy), Reporte general por clases y cálculo/representación de la Matriz de Confusión.
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Configuración del estilo visual de las gráficas (utiliza un estilo limpio con rejilla si está disponible en la versión).
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# Mensaje de confirmación de carga correcta del entorno.
print('✅ Entorno y librerías de Scikit-Learn cargados correctamente.')

**Doble naturaleza del dataset:** En esta práctica importamos dos conjuntos de datos de naturaleza distinta:

1.California Housing (donde la variable objetivo $y$ es un valor continuo en dólares, $y \in \mathbb{R}$)

2.Breast Cancer (donde la variable objetivo $y$ es una etiqueta discreta binaria, $y \in \{0, 1\}$).

**Paralelismo SVC vs. SVR:**

Scikit-Learn utiliza la sintaxis SVC para clasificación (Support Vector Classifier) y SVR para regresión (Support Vector Regressor), aplicando los mismos principios geométricos de margen máximo (Punto 2.4.1 de Marcombo).

**Estandarización obligatoria (StandardScaler):**

El escalado no es opcional en SVM, ya que la optimización del hiperplano calcula distancias euclídeas directas en el espacio de características.

## Bloque 02: Regresión Lineal (Simple y Múltiple)

*(Referencia Marcombo: Unidad 2 / Punto 2.4.2)*

En los problemas de **regresión**, la variable objetivo $y$ es un valor numérico continuo. El algoritmo busca ajustar la mejor recta o hiperplano que minimice la suma de errores cuadráticos ($MSE$):

$$\hat{y} = w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b$$

In [ ]:
# =====================================================================
# EJEMPLO GUIADO: Regresión Lineal Múltiple en California Housing
# (Referencia Marcombo: Unidad 2 / Punto 2.4.2)
# =====================================================================

import pprint;

# Carga del dataset de regresión en formato DataFrame de Pandas (as_frame=True)
# El parámetro as_frame=True en las funciones de carga de datasets de Scikit-Learn
# le indica a la librería que devuelva las estructuras de datos en formato Pandas (DataFrames y Series)
# en lugar de matrices numéricas simples de NumPy.
housing = fetch_california_housing(as_frame=True)

# Seleccionamos una matriz de características (X_reg) con 4 variables predictoras continuas:
# - MedInc: Ingreso medio de los bloques de viviendas.
# - AveRooms: Promedio de habitaciones por vivienda.
# - HouseAge: Antigüedad mediana de las casas.
# - AveOccup: Promedio de ocupantes por vivienda.
X_reg = housing.data[['MedInc', 'AveRooms', 'HouseAge', 'AveOccup']]

# Definimos la variable objetivo continua (y_reg), que representa el valor medio de la vivienda en cientos de miles de dólares ($100.000s).
y_reg = housing.target

# Partición representativa del conjunto de datos en un 80% para Entrenamiento (Train) y un 20% para Test (Evaluación).
# Fijamos 'random_state=42' para garantizar que la división sea idéntica en cada ejecución (reproducibilidad).
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=42
)

# 1. INSTANCIACIÓN Y ENTRENAMIENTO DEL MODELO DE REGRESIÓN LINEAL MÚLTIPLE
# Creamos el objeto de Regresión Lineal basado en la optimización analítica por Mínimos Cuadrados Ordinarios.
model_lr = LinearRegression()

# Ajustamos el modelo ajustando la ecuación y = w1*x1 + w2*x2 + ... + b sobre el conjunto de entrenamiento.
model_lr.fit(X_train_reg, y_train_reg)

# 2. FASE DE PREDICCIÓN (INFERENCIA)
# Evaluamos el modelo calculando los valores continuos predichos (y_pred_reg) sobre las muestras reservadas de test (X_test_reg).
y_pred_reg = model_lr.predict(X_test_reg)

# 3. CÁLCULO DE MÉTRICAS FORMALES DE ERROR DE REGRESIÓN
# Error Cuadrático Medio (MSE): Penaliza fuertemente los errores grandes al elevar las desviaciones al cuadrado.
mse = mean_squared_error(y_test_reg, y_pred_reg)

# Raíz del Error Cuadrático Medio (RMSE): Expresa el error medio en las mismas unidades que la variable objetivo ($100.000s).
rmse = np.sqrt(mse)

# Error Absoluto Medio (MAE): Promedio directo en valor absoluto de los desvíos entre los valores reales y los predichos.
mae = mean_absolute_error(y_test_reg, y_pred_reg)

# Coeficiente de Determinación (R²): Proporción de la varianza total explicada por el modelo (1.0 = ajuste perfecto, 0.0 = modelo base).
r2 = r2_score(y_test_reg, y_pred_reg)

# Impresión ordenada de los pesos del hiperplano y las métricas de rendimiento en consola.
print('=== EVALUACIÓN DEL MODELO DE REGRESIÓN LINEAL MÚLTIPLE ===')
print(f'Ecuación del hiperplano: Pendientes (w) = {model_lr.coef_} | Intersección (b) = {model_lr.intercept_:.4f}')
print(f'📉 Error Cuadrático Medio (MSE): {mse:.4f}')
print(f'📉 Raíz del Error Cuadrático Medio (RMSE): {rmse:.4f}')
print(f'📉 Error Absoluto Medio (MAE): {mae:.4f}')
print(f'📊 Coeficiente de Determinación (R²): {r2:.4f}')
#pprint.pprint(model_lr.__dict__)

In [ ]:
# =====================================================================
# INSPECCIÓN DE LA FICHA TÉCNICA DEL DATASET (California Housing)
# =====================================================================

# El objeto 'housing' cargado mediante fetch_california_housing es una estructura Bunch.
# Contiene el atributo '.DESCR' con la documentación oficial del censo original de 1990.

# Imprimimos el texto descriptivo utilizando print() para interpretar correctamente los saltos de línea (\n)
print("=================================================================")
print("          FICHA TÉCNICA Y DOCUMENTACIÓN DEL DATASET              ")
print("=================================================================\n")
print(housing.DESCR)

En este reto, debeís construir un modelo de Regresión Lineal Simple utilizando únicamente la variable de mayor correlación (MedInc: Ingreso Medio) para comparar la recta resultante frente al hiperplano múltiple anterior.

In [ ]:
# =====================================================================
# EJERCICIO PRÁCTICO: Regresión Lineal Simple (Univariable)
# =====================================================================

# Seleccionamos únicamente la variable 'MedInc' como matriz unidimensional de características (X_simple)
X_simple = housing.data[['MedInc']]

# Partición del dataset reservando el 20% para test
X_train_sim, X_test_sim, y_train_sim, y_test_sim = train_test_split(
    X_simple, y_reg, test_size=0.20, random_state=42
)

# 1. Instancia el modelo de Regresión Lineal Simple
model_simple = # TU CÓDIGO AQUÍ

# 2. Entrena el modelo utilizando los datos de entrenamiento simples (X_train_sim, y_train_sim)
# TU CÓDIGO AQUÍ

# 3. Realiza las predicciones sobre el conjunto de test (X_test_sim) y guárdalas en 'y_pred_sim'
y_pred_sim = # TU CÓDIGO AQUÍ

# 4. Calcula la Raíz del Error Cuadrático Medio (RMSE) y el Coeficiente de Determinación (R²)
rmse_sim = # TU CÓDIGO AQUÍ (puedes usar np.sqrt(mean_squared_error(...)))
r2_sim = # TU CÓDIGO AQUÍ

print('=== EVALUACIÓN DEL MODELO DE REGRESIÓN LINEAL SIMPLE ===')
print(f'📉 Raíz del Error Cuadrático Medio (RMSE): {rmse_sim:.4f}')
print(f'📊 Coeficiente de Determinación (R²): {r2_sim:.4f}')

## Bloque 03: Regresión Logística (Clasificación Categórica Probabilística)

*(Referencia Marcombo: Unidad 2, Pág. 29-30 / Punto 2.4.1 y Figuras 2.16-2.17)*

A pesar de su nombre, la **Regresión Logística** es un algoritmo de **clasificación**. Aplica la función sigmoide/logit $\sigma(z) = \frac{1}{1 + e^{-z}}$ sobre una combinación lineal para transformar cualquier valor en una probabilidad acotada entre $0$ y $1$.

In [ ]:
# =====================================================================
# EJEMPLO GUIADO: Regresión Logística en Clasificación Binaria
# (Referencia Marcombo: Unidad 2, Pág. 29 / Punto 2.4.1)
# =====================================================================

# 1. CARGA Y PREPARACIÓN DEL DATASET BIOMÉDICO
# Cargamos el conjunto de datos de Cáncer de Mama de Scikit-Learn (30 atributos físicos de tumores).
# Las etiquetas objetivo 'y_clf' son binarias: 0 = Maligno, 1 = Benigno.
cancer = load_breast_cancer()
X_clf, y_clf = cancer.data, cancer.target


# 2. PARTICIÓN ESTRATIFICADA DE DATOS (TRAIN / TEST)
# Dividimos las observaciones en un 70% para Entrenamiento (Train) y un 30% para Test (Evaluación).
# Usamos 'stratify=y_clf' para asegurar que la proporción de tumores benignos/malignos sea idéntica
# tanto en el conjunto de entrenamiento como en el conjunto de prueba.
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.30, random_state=42, stratify=y_clf
)


# 3. ESTANDARIZACIÓN Y PREVENCIÓN DE FUGA DE DATOS (DATA LEAKAGE)
# Creamos el objeto escalador Z-score para normalizar la escala de las 30 variables continuas.
scaler = StandardScaler()

# Calculamos la media y varianza SOLO sobre los datos de entrenamiento y los transformamos (.fit_transform).
X_train_clf_scaled = scaler.fit_transform(X_train_clf)

# Aplicamos la misma transformación sobre los datos de test usando la media de entrenamiento (.transform).
X_test_clf_scaled = scaler.transform(X_test_clf)


# 4. INSTANCIACIÓN Y ENTRENAMIENTO DE LA REGRESIÓN LOGÍSTICA
# Creamos el modelo de Regresión Logística fijando 'random_state=42' para asegurar reproducibilidad.
# Este algoritmo aplica la función sigmoide/logit para mapear combinaciones lineales a probabilidades entre 0 y 1.
model_logreg = LogisticRegression(random_state=42)

# Ajustamos el modelo pasando la matriz de entrenamiento escalada (X_train_clf_scaled) y las clases reales (y_train_clf).
model_logreg.fit(X_train_clf_scaled, y_train_clf)


# 5. FASE DE PREDICCIÓN (INFERENCIA DISCRETA Y PROBABILÍSTICA)
# Generamos la predicción binaria final (0 o 1) aplicando el umbral estándar por defecto del 50% (p >= 0.5).
y_pred_logreg = model_logreg.predict(X_test_clf_scaled)

# Extraemos la probabilidad continua de pertenecer a la clase positiva (Clase 1 = Benigno).
# Tomamos la segunda columna [:, 1] del array de probabilidades devueltas por el método '.predict_proba()'.
y_prob_logreg = model_logreg.predict_proba(X_test_clf_scaled)[:, 1]


# 6. EVALUACIÓN DEL RENDIMIENTO (ACCURACY)
# Calculamos el porcentaje global de acierto comparando las etiquetas reales (y_test_clf) con las predicciones (y_pred_logreg).
acc_logreg = accuracy_score(y_test_clf, y_pred_logreg)

# Imprimimos la métrica final expresada en porcentaje.
print(f'🎯 Regresión Logística - Accuracy en Test: {acc_logreg * 100:.2f}%')

###💡 Claves Didácticas para Destacar en el Aula
**1.La importancia de stratify=y_clf:** En problemas biomédicos o bancarios (donde las clases pueden estar desequilibradas), es crítico forzar a train_test_split a mantener el mismo porcentaje de la clase positiva en train y en test.

**2.Diferencia entre .fit_transform() y .transform():**

* Sobre X_train se usa .fit_transform() para que el escalador "aprenda" la media y la desviación típica del conjunto de entrenamiento.
* Sobre X_test se usa únicamente .transform() para evitar la fuga de información (Data Leakage) del futuro hacia el pasado.

**3.Predicción vs. Probabilidad:**
Observemos la diferencia entre .predict() (que devuelve la decisión binaria directa: 0 o 1) y .predict_proba() (que devuelve la estimación continua entre 0.0 y 1.0), conceptualizando la función Sigmoide vista en la teoría de la Unidad 2 de Marcombo (Pág. 30).

In [ ]:
# =====================================================================
# EJERCICIO PRÁCTICO: Regresión Logística Simplificada (2 Variables)
# (Referencia Marcombo: Unidad 2, Punto 2.4.1)
# =====================================================================

# OBJETIVO: Entrenar un modelo de Regresión Logística utilizando únicamente
# las 2 primeras características del tumores ('Radio Medio' y 'Textura Media')
# para simplificar el modelo y analizar cómo varía la exactitud obtenida.

# 1. SELECCIÓN DE CARACTERÍSTICAS Y PREPARACIÓN DE DATOS
# Seleccionamos las dos primeras columnas de la matriz X_clf (índices 0 y 1)
X_clf_2var = X_clf[:, :2]

# Partición estratificada (70% Train / 30% Test) con las 2 variables seleccionadas
X_train_2v, X_test_2v, y_train_2v, y_test_2v = train_test_split(
    X_clf_2var, y_clf, test_size=0.30, random_state=42, stratify=y_clf
)

# Estandarización de las 2 variables continuas mediante StandardScaler
scaler_2v = StandardScaler()
X_train_2v_scaled = scaler_2v.fit_transform(X_train_2v)
X_test_2v_scaled = scaler_2v.transform(X_test_2v)


# 2. INSTANCIACIÓN DEL MODELO LOGÍSTICO
# Crea el objeto de Regresión Logística fijando 'random_state=42' para reproducibilidad.
model_logreg_2v = # TU CÓDIGO AQUÍ


# 3. FASE DE ENTRENAMIENTO
# Ajusta el modelo llamando a '.fit()' sobre los datos de entrenamiento escalados (X_train_2v_scaled, y_train_2v).
# TU CÓDIGO AQUÍ


# 4. FASE DE PREDICCIÓN (INFERENCIA DISCRETA Y PROBABILÍSTICA)
# Genera las predicciones de clase binarias (0 o 1) sobre el conjunto de test escalado (X_test_2v_scaled)
y_pred_2v = # TU CÓDIGO AQUÍ

# Extrae la probabilidad continua sigmoidea de pertenecer a la Clase 1 (Benigno) usando '.predict_proba(...)[:, 1]'
y_prob_2v = # TU CÓDIGO AQUÍ


# 5. EVALUACIÓN Y COMPARATIVA DE ACCURACY
# Calcula la exactitud (Accuracy) comparando 'y_test_2v' con las predicciones 'y_pred_2v'
acc_logreg_2v = # TU CÓDIGO AQUÍ

# Mostramos el resultado y lo comparamos con el modelo complejo de 30 variables
print(f"🎯 Regresión Logística (2 variables) - Accuracy: {acc_logreg_2v * 100:.2f}%")
print(f"📊 Comparativa: Modelo Completo 30 vars ({acc_logreg * 100:.2f}%) vs Modelo Simplificado 2 vars ({acc_logreg_2v * 100:.2f}%)")

## Bloque 04: Máquinas de Vectores de Soporte (SVM)

*(Referencia Marcombo: Unidad 2, Pág. 31-32 / Punto 2.4.1 y Figuras 2.18-2.19)*

Las **SVM** buscan el **hiperplano de margen máximo** que maximiza la distancia geométrica entre las clases. Utilizan el parámetro $C$ (penalización de errores) y funciones *kernel* (lineal, RBF, polinomial) para proyectar datos no separables a dimensiones superiores.

In [ ]:
# =====================================================================
# EJEMPLO GUIADO: Comparativa de Kernels en SVM (Lineal vs. RBF)
# (Referencia Marcombo: Unidad 2, Págs. 29-32 / Punto 2.4.1)
# =====================================================================

# ---------------------------------------------------------------------
# 1. MÁQUINA DE VECTORES DE SOPORTE CON KERNEL LINEAL
# ---------------------------------------------------------------------
# Instanciamos el clasificador SVC con 'kernel="linear"':
# Busca el hiperplano plano de separación de margen máximo en el espacio original de las variables.
# 'C=1.0' controla el margen blando (penalización de errores) y 'random_state=42' fija la semilla.
svm_linear = SVC(kernel='linear', C=1.0, random_state=42)

# Entrenamos el modelo lineal pasando los datos de entrenamiento previamente estandarizados con Z-score.
svm_linear.fit(X_train_clf_scaled, y_train_clf)

# Realizamos las predicciones de clase binaria (0=Maligno, 1=Benigno) sobre el conjunto de test escalado.
y_pred_svm_lin = svm_linear.predict(X_test_clf_scaled)

# Calculamos la exactitud (Accuracy) del modelo con kernel lineal comparando predicciones con clases reales.
acc_svm_lin = accuracy_score(y_test_clf, y_pred_svm_lin)


# ---------------------------------------------------------------------
# 2. MÁQUINA DE VECTORES DE SOPORTE CON KERNEL NO LINEAL (RBF)
# ---------------------------------------------------------------------
# Instanciamos el clasificador SVC con 'kernel="rbf"' (Radial Basis Function / Función de Base Gaussiana):
# Aplica el "Truco del Kernel" proyectando los datos a un espacio de mayor dimensión donde sean separables linealmente.
# 'gamma="scale"' ajusta automáticamente el radio de influencia de los vectores de soporte según la varianza del dataset.
svm_rbf = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)

# Entrenamos el modelo no lineal sobre el conjunto de entrenamiento escalado.
svm_rbf.fit(X_train_clf_scaled, y_train_clf)

# Generamos las predicciones categóricas sobre el conjunto de test reservado.
y_pred_svm_rbf = svm_rbf.predict(X_test_clf_scaled)

# Calculamos la exactitud (Accuracy) del modelo RBF.
acc_svm_rbf = accuracy_score(y_test_clf, y_pred_svm_rbf)


# ---------------------------------------------------------------------
# 3. IMPRESIÓN DE RESULTADOS Y ANÁLISIS DE VECTORES DE SOPORTE
# ---------------------------------------------------------------------
# La propiedad '.support_' contiene los índices de los puntos del conjunto de entrenamiento que forman los límites de la calle (margen).
# Medimos la longitud de dicho atributo con len() para saber cuántas muestras actúan como vectores de soporte.
print(f'🎯 SVM Kernel Lineal - Accuracy: {acc_svm_lin * 100:.2f}% | Vectores de soporte: {len(svm_linear.support_)}')
print(f'🎯 SVM Kernel RBF   - Accuracy: {acc_svm_rbf * 100:.2f}% | Vectores de soporte: {len(svm_rbf.support_)}')

Antes de abordar el ejercicio práctico, revisa estos 4 conceptos fundamentales observados en el código anterior:

1. **Obligatoriedad del Escalado ($Z$-score / `StandardScaler`):**  
   Las SVM construyen un hiperplano buscando la **máxima distancia geométrica (margen)** a los puntos más cercanos. Si una variable tiene valores del orden de $1000$ (como el área) y otra del orden de $0.08$ (como la suavidad), la primera dominará por completo el cálculo del margen. **Jamás entrenes una SVM sin estandarizar previamente las variables (`X_train_clf_scaled`)**.

2. **El "Truco del Kernel" (*Kernel Trick*):**  
   * **Kernel Lineal (`kernel='linear'`):** Intenta separar las clases trazando una línea o hiperplano plano en el espacio original. Funciona bien si las características guardan una relación de separación lineal simple.
   * **Kernel RBF (`kernel='rbf'`):** Proyecta de forma implícita los datos a un espacio de dimensión superior utilizando funciones gaussianas. Permite trazar fronteras de decisión curvas y complejas cuando los datos no son separables en su dimensión original.

3. **Análisis de los Vectores de Soporte (`len(model.support_)`):**  
   Los **vectores de soporte** son las únicas muestras del dataset de entrenamiento situadas justo en los bordes de la "calle" o margen de seguridad. El resto de datos le resultan irrelevantes al modelo.  
   * **Interpretación:** Un número excesivamente alto de vectores de soporte suele indicar que la frontera de decisión es demasiado compleja o sufre riesgo de **sobreajuste (*overfitting*)**.

4. **El Parámetro de Amplitud Gaussiana (`gamma='scale'`):**  
   En el Kernel RBF, `gamma` define el radio de influencia de cada vector de soporte. Al fijar `gamma='scale'`, Scikit-Learn calcula automáticamente el radio óptimo en función de la varianza del dataset, evitando que las curvas sean ni demasiado cerradas (sobreajuste) ni demasiado suaves (subajuste).

In [ ]:
# =====================================================================
# EJERCICIO PRÁCTICO: Exploración de Kernels y Regularización (C) en SVM
# (Referencia Marcombo: Unidad 2, Págs. 29-32 / Punto 2.4.1)
# =====================================================================

# ---------------------------------------------------------------------
# RETO 1: MÁQUINA DE VECTORES DE SOPORTE CON KERNEL POLINÓMICO
# ---------------------------------------------------------------------
# Instancia un clasificador SVC utilizando 'kernel="poly"', grado 3 ('degree=3'),
# 'C=1.0' y semilla aleatoria 'random_state=42'.
# Este kernel proyecta los datos mediante combinaciones polinómicas de las variables.
svm_poly = # TU CÓDIGO AQUÍ

# Entrena el modelo polinómico utilizando el conjunto de datos de entrenamiento escalado (X_train_clf_scaled, y_train_clf).
# TU CÓDIGO AQUÍ

# Genera las predicciones categóricas sobre el conjunto de test escalado (X_test_clf_scaled).
y_pred_svm_poly = # TU CÓDIGO AQUÍ

# Calcula la exactitud (Accuracy) del modelo polinómico evaluando predicciones frente a etiquetas reales (y_test_clf).
acc_svm_poly = # TU CÓDIGO AQUÍ


# ---------------------------------------------------------------------
# RETO 2: IMPACTO DE UNA ALTA PENALIZACIÓN DE MARGEN (C = 100.0)
# ---------------------------------------------------------------------
# Instancia un clasificador SVC con 'kernel="rbf"', pero esta vez aumenta el parámetro
# de regularización a 'C=100.0' para forzar un margen mucho más estricto (penaliza más los errores).
# Manten 'gamma="scale"' y 'random_state=42'.
svm_rbf_c100 = # TU CÓDIGO AQUÍ

# Entrena el modelo sobre el conjunto de entrenamiento escalado.
# TU CÓDIGO AQUÍ

# Genera las predicciones sobre el conjunto de test reservado.
y_pred_svm_c100 = # TU CÓDIGO AQUÍ

# Calcula la exactitud (Accuracy) de este modelo con C elevado.
acc_svm_c100 = # TU CÓDIGO AQUÍ


# ---------------------------------------------------------------------
# 3. COMPARATIVA Y ANÁLISIS DE RESULTADOS
# ---------------------------------------------------------------------
print(f'🎯 SVM Kernel Polinómico (Grado 3) - Accuracy: {acc_svm_poly * 100:.2f}% | Vectores de soporte: {len(svm_poly.support_)}')
print(f'🎯 SVM Kernel RBF (C=100.0)         - Accuracy: {acc_svm_c100 * 100:.2f}% | Vectores de soporte: {len(svm_rbf_c100.support_)}')

## Bloque 05: Cuestionario de Experimentación Guiada para el Alumno

Resuelve las siguientes tareas rellenando el código indicado:

### Reto 1: Experimento de Regularización $C$ en SVM
Este código modifica el parámetro de regularización $C$ en la SVM con Kernel RBF ($C=0.01$, $C=1.0$ y $C=100.0$). ¿Cómo afecta a la cantidad de vectores de soporte y a la precisión?

Lee y comenta el código tratndo de explicar, que ocurre en cada línea.

In [ ]:

valores_C = [0.01, 1.0, 100.0]
resultados_svm = []

for c in valores_C:
    model_svm = SVC(kernel='rbf', C=c, random_state=42)
    model_svm.fit(X_train_clf_scaled, y_train_clf)
    preds = model_svm.predict(X_test_clf_scaled)
    acc = accuracy_score(y_test_clf, preds)
    resultados_svm.append({'C': c, 'Accuracy': acc, 'Vectores_Soporte': len(model_svm.support_)})

df_res_svm = pd.DataFrame(resultados_svm)
print('--- IMPACTO DEL HIPERPARÁMETRO C EN SVM ---')
print(df_res_svm)

In [ ]:
# =====================================================================
# SOLUCIÓN: Barrido Automatizado del Hiperparámetro C en SVM
# (Referencia Marcombo: Unidad 2, Págs. 29-32 / Punto 2.4.1)
# =====================================================================

# 1. DEFINICIÓN DE LA MESH/LISTA DE VALORES A EVALUAR
# Definimos una lista con 3 magnitudes del hiperparámetro de regularización 'C':
# - C = 0.01  : Margen muy "blando" (permite muchos errores en train, busca máxima simplicidad/generalización).
# - C = 1.0   : Valor estándar por defecto en Scikit-Learn (equilibrio entre margen y penalización de errores).
# - C = 100.0 : Margen muy "duro" o estricto (penaliza fuertemente los errores en train, riesgo de sobreajuste).
valores_C = [0.01, 1.0, 100.0]

# Inicializamos una lista vacía para almacenar los diccionarios con las métricas de cada iteración.
resultados_svm = []

# 2. BUCLE DE EXPERIMENTACIÓN (FOR LOOP)
# Iteramos secuencialmente sobre cada uno de los valores definidos en la lista 'valores_C'.
for c in valores_C:

    # Instanciamos el clasificador SVC con kernel RBF (no lineal), asignando el valor actual del bucle a 'C'.
    # Fijamos 'random_state=42' para garantizar que el experimento sea totalmente reproducible.
    model_svm = SVC(kernel='rbf', C=c, random_state=42)

    # Entrenamos el modelo pasando la matriz de entrenamiento estandarizada (Z-score) y las clases reales.
    model_svm.fit(X_train_clf_scaled, y_train_clf)

    # Generamos las predicciones discretas de clase (0=Maligno, 1=Benigno) sobre el conjunto de test escalado.
    preds = model_svm.predict(X_test_clf_scaled)

    # Calculamos la exactitud (Accuracy) del modelo comparando las etiquetas de test con las predicciones.
    acc = accuracy_score(y_test_clf, preds)

    # Guardamos los resultados del experimento actual en un diccionario y lo añadimos a la lista 'resultados_svm'.
    # Inspeccionamos la longitud de '.support_' (len(model_svm.support_)) para contar cuántos vectores de soporte necesitó el modelo.
    resultados_svm.append({
        'C': c,
        'Accuracy': acc,
        'Vectores_Soporte': len(model_svm.support_)
    })

# 3. EMBALAJE Y PRESENTACIÓN DE RESULTADOS EN PANDAS
# Convertimos la lista de diccionarios acumulados en un DataFrame estructurado para una lectura clara de los datos.
df_res_svm = pd.DataFrame(resultados_svm)

# Imprimimos la cabecera y el cuadro comparativo final por consola.
print('--- IMPACTO DEL HIPERPARÁMETRO C EN SVM ---')
print(df_res_svm)

## Wrap-Up: Consolidación de Conceptos (Práctica 2)

1. **Regresión vs. Clasificación:** Aprendimos a seleccionar la función de pérdida adecuada según el tipo de variable objetivo (numérica continua vs. categórica discreta).
2. **Escalado Estandarizado:** Validamos que modelos basados en margenes geométricos (SVM) o gradiente descendente (Regresión Logística) requieren **estandarización previa** ($Z-score$).
3. **Regularización ($C$):** Comprobamos que un valor de $C$ muy bajo puede producir subajuste (*underfitting*), mientras que un $C$ excesivamente alto fuerza un margen estrecho y aumenta el riesgo de sobreajuste (*overfitting*).

#SOLUCIONES

In [ ]:
#@title  SOLUCIÓN: Regresión Lineal Simple
# =====================================================================

# Preparación previa de datos (ya incluida en la plantilla)
X_simple = housing.data[['MedInc']]

X_train_sim, X_test_sim, y_train_sim, y_test_sim = train_test_split(
    X_simple, y_reg, test_size=0.20, random_state=42
)

# ---------------------------------------------------------------------
# 1. INSTANCIACIÓN DEL MODELO
# ---------------------------------------------------------------------
# Creamos el objeto de Regresión Lineal de Scikit-Learn.
# Ajustará la ecuación de la recta: y = w*x + b (donde x es 'MedInc')
model_simple = LinearRegression()

# ---------------------------------------------------------------------
# 2. ENTRENAMIENTO DEL MODELO
# ---------------------------------------------------------------------
# Calculamos los parámetros (pendiente w e intersección b) minimizando
# la suma de errores al cuadrado sobre los datos de entrenamiento.
model_simple.fit(X_train_sim, y_train_sim)

# ---------------------------------------------------------------------
# 3. FASE DE PREDICCIÓN (INFERENCIA)
# ---------------------------------------------------------------------
# Evaluamos el modelo generando las predicciones de precio estimado ($100.000s)
# sobre el 20% de datos reservado para test.
y_pred_sim = model_simple.predict(X_test_sim)

# ---------------------------------------------------------------------
# 4. EVALUACIÓN DE MÉTRICAS (RMSE y R²)
# ---------------------------------------------------------------------
# RMSE (Root Mean Squared Error): Mide el error promedio de predicción en la unidad original ($100.000s).
rmse_sim = np.sqrt(mean_squared_error(y_test_sim, y_pred_sim))

# R² (Coeficiente de Determinación): Indica el porcentaje de varianza del precio explicada por 'MedInc'.
r2_sim = r2_score(y_test_sim, y_pred_sim)

# ---------------------------------------------------------------------
# 5. IMPRESIÓN DE RESULTADOS
# ---------------------------------------------------------------------
print('=== EVALUACIÓN DEL MODELO DE REGRESIÓN LINEAL SIMPLE ===')
print(f'📉 Raíz del Error Cuadrático Medio (RMSE): {rmse_sim:.4f}')
print(f'📊 Coeficiente de Determinación (R²): {r2_sim:.4f}')

In [ ]:
#@title SOLUCIÓN: Regresión Logística Simplificada
# (Referencia Marcombo: Unidad 2, Punto 2.4.1)
# =====================================================================

# OBJETIVO: Entrenar un modelo de Regresión Logística utilizando únicamente
# las 2 primeras características de los tumores ('Radio Medio' y 'Textura Media')
# para simplificar el modelo y analizar cómo varía la exactitud obtenida.

# 1. SELECCIÓN DE CARACTERÍSTICAS Y PREPARACIÓN DE DATOS
# Seleccionamos las dos primeras columnas de la matriz X_clf (índices 0 y 1)
X_clf_2var = X_clf[:, :2]

# Partición estratificada (70% Train / 30% Test) con las 2 variables seleccionadas
X_train_2v, X_test_2v, y_train_2v, y_test_2v = train_test_split(
    X_clf_2var, y_clf, test_size=0.30, random_state=42, stratify=y_clf
)

# Estandarización de las 2 variables continuas mediante StandardScaler
scaler_2v = StandardScaler()
X_train_2v_scaled = scaler_2v.fit_transform(X_train_2v)
X_test_2v_scaled = scaler_2v.transform(X_test_2v)


# 2. INSTANCIACIÓN DEL MODELO LOGÍSTICO
# Creamos el objeto de Regresión Logística fijando 'random_state=42' para asegurar reproducibilidad.
model_logreg_2v = LogisticRegression(random_state=42)


# 3. FASE DE ENTRENAMIENTO
# Entrenamos el modelo pasando la matriz de 2 variables escalada y las etiquetas reales de entrenamiento.
model_logreg_2v.fit(X_train_2v_scaled, y_train_2v)


# 4. FASE DE PREDICCIÓN (INFERENCIA DISCRETA Y PROBABILÍSTICA)
# Generamos la predicción binaria (0 o 1) sobre las observaciones de test escaladas.
y_pred_2v = model_logreg_2v.predict(X_test_2v_scaled)

# Extraemos la probabilidad continua sigmoidea de pertenecer a la clase 1 (Benigno) usando la segunda columna [:, 1].
y_prob_2v = model_logreg_2v.predict_proba(X_test_2v_scaled)[:, 1]


# 5. EVALUACIÓN Y COMPARATIVA DE ACCURACY
# Calculamos la exactitud (Accuracy) comparando las etiquetas de test reales con las predicciones.
acc_logreg_2v = accuracy_score(y_test_2v, y_pred_2v)

# Mostramos el resultado final y lo comparamos con el modelo de 30 variables.
print(f"🎯 Regresión Logística (2 variables) - Accuracy: {acc_logreg_2v * 100:.2f}%")
print(f"📊 Comparativa: Modelo Completo 30 vars ({acc_logreg * 100:.2f}%) vs Modelo Simplificado 2 vars ({acc_logreg_2v * 100:.2f}%)")

In [ ]:
#@title SOLUCIÓN: Exploración de Kernels y Regularización (C) en SVM
# (Referencia Marcombo: Unidad 2, Págs. 29-32 / Punto 2.4.1)
# =====================================================================

# ---------------------------------------------------------------------
# RETO 1: MÁQUINA DE VECTORES DE SOPORTE CON KERNEL POLINÓMICO
# ---------------------------------------------------------------------
# Instanciamos el clasificador SVC con kernel polinómico de grado 3 ('degree=3'),
# C=1.0 y fijamos la semilla aleatoria en 42 para garantizar la reproducibilidad.
svm_poly = SVC(kernel='poly', degree=3, C=1.0, random_state=42)

# Entrenamos el modelo polinómico utilizando el conjunto de datos de entrenamiento escalado.
svm_poly.fit(X_train_clf_scaled, y_train_clf)

# Generamos las predicciones categóricas sobre el conjunto de test escalado.
y_pred_svm_poly = svm_poly.predict(X_test_clf_scaled)

# Calculamos la exactitud (Accuracy) evaluando las predicciones frente a las etiquetas reales.
acc_svm_poly = accuracy_score(y_test_clf, y_pred_svm_poly)


# ---------------------------------------------------------------------
# RETO 2: IMPACTO DE UNA ALTA PENALIZACIÓN DE MARGEN (C = 100.0)
# ---------------------------------------------------------------------
# Instanciamos el clasificador SVC con kernel RBF aumentando la penalización a C=100.0.
# Un valor alto de C fuerza un margen más estricto y penaliza fuertemente los errores de clasificación.
svm_rbf_c100 = SVC(kernel='rbf', C=100.0, gamma='scale', random_state=42)

# Entrenamos el modelo sobre el conjunto de entrenamiento escalado.
svm_rbf_c100.fit(X_train_clf_scaled, y_train_clf)

# Generamos las predicciones sobre el conjunto de test reservado.
y_pred_svm_c100 = svm_rbf_c100.predict(X_test_clf_scaled)

# Calculamos la exactitud (Accuracy) de este modelo con C elevado.
acc_svm_c100 = accuracy_score(y_test_clf, y_pred_svm_c100)


# ---------------------------------------------------------------------
# 3. COMPARATIVA Y ANÁLISIS DE RESULTADOS
# ---------------------------------------------------------------------
print(f'🎯 SVM Kernel Polinómico (Grado 3) - Accuracy: {acc_svm_poly * 100:.2f}% | Vectores de soporte: {len(svm_poly.support_)}')
print(f'🎯 SVM Kernel RBF (C=100.0)         - Accuracy: {acc_svm_c100 * 100:.2f}% | Vectores de soporte: {len(svm_rbf_c100.support_)}')